In [1]:
import pandas as pd
import os
os.environ['USE_PYGEOS'] = '0'
import geopandas as gpd
from shapely.geometry import Point
import numpy as np

In [4]:
def get_rooftop_pv_cf_meta():
    rooftop_pv_cf_meta = pd.read_csv("../data/distpv_profiles/county_centroid_project_points_conus.csv")
    rooftop_pv_cf_meta['FIPS'] = (
        'p' + rooftop_pv_cf_meta['GEOID'].astype(str).str.zfill(5)
    )
    rooftop_pv_cf_meta_geometry = [
        Point(xy) for xy in zip(rooftop_pv_cf_meta['longitude'], rooftop_pv_cf_meta['latitude'])
    ]
    rooftop_pv_cf_meta = gpd.GeoDataFrame(
        rooftop_pv_cf_meta,
        geometry=rooftop_pv_cf_meta_geometry,
        crs='EPSG:4326'
    )

    return rooftop_pv_cf_meta

In [6]:
def create_gid_county_map(rooftop_pv_cf_meta):
    ## Create mapping between counties and rooftop PV profile GIDs
    county_centroids = gpd.read_file("../data/shapefiles/US_COUNTY_2022")
    county_centroids['geometry'] = county_centroids['geometry'].centroid

    rooftop_pv_cf_meta_matched = rooftop_pv_cf_meta.loc[rooftop_pv_cf_meta.FIPS.isin(county_centroids.rb)]
    
    rooftop_pv_cf_meta_unmatched = rooftop_pv_cf_meta.loc[~rooftop_pv_cf_meta.FIPS.isin(county_centroids.rb)]
    county_centroids_unmatched = county_centroids.loc[~county_centroids.rb.isin(rooftop_pv_cf_meta_matched.FIPS)]
    rooftop_pv_cf_meta_unmatched = (
        gpd.sjoin_nearest(
            county_centroids_unmatched.drop(columns='FIPS').rename(columns={'rb': 'FIPS'})[['FIPS', 'geometry']],
            rooftop_pv_cf_meta_unmatched.drop(columns='FIPS').to_crs(county_centroids.crs),
            how='left'
        )
        .to_crs(rooftop_pv_cf_meta.crs)
        .drop(columns='index_right')
    )
    
    gid_county_map = (
        pd.concat([
            rooftop_pv_cf_meta_matched,
            rooftop_pv_cf_meta_unmatched
        ])
        .groupby('gid')
        ['FIPS']
        .apply(list)
        .explode()
    )

    return gid_county_map

In [7]:
def get_rooftop_pv_cf_profile(sector, gid_county_map):
    # Get rooftop PV CF profiles and normalize
    rooftop_pv_cf = pd.read_csv(f"../data/distpv_profiles/county_level_distpv_profiles_{sector}_2007_2023.csv")
    rooftop_pv_cf["datetime"] = pd.to_datetime(rooftop_pv_cf["Unnamed: 0"].str.split("\'").str[1])
    rooftop_pv_cf = rooftop_pv_cf.set_index("datetime").drop(columns=["Unnamed: 0"])
    rooftop_pv_cf = rooftop_pv_cf / rooftop_pv_cf.max().max()
    
    for year in [2008, 2012, 2016, 2020]:
        df = (
            rooftop_pv_cf.loc[rooftop_pv_cf.index.year == year]
            .copy()
            .tail(24)
        )
        df = df.set_index(df.index.map(lambda x: x.replace(day=31)))
        rooftop_pv_cf = pd.concat([rooftop_pv_cf, df])
    
    rooftop_pv_cf = rooftop_pv_cf.sort_index()
    
    # Add county information
    rooftop_pv_cf.columns = rooftop_pv_cf_meta.gid
    rooftop_pv_cf = (
        rooftop_pv_cf.transpose()
        .merge(gid_county_map, left_index=True, right_index=True)
        .set_index('FIPS')
        .transpose()
    )
    
    rooftop_pv_cf.columns.name = ''
    rooftop_pv_cf.index.names = ['timestamp']
    rooftop_pv_cf.index = pd.to_datetime(rooftop_pv_cf.index)
    
    rooftop_pv_cf_2024 = rooftop_pv_cf.iloc[-24:].copy()
    rooftop_pv_cf_2024.index = rooftop_pv_cf_2024.index + pd.Timedelta(days=1)
    rooftop_pv_cf = pd.concat([rooftop_pv_cf, rooftop_pv_cf_2024])
    
    # Profiles start in UTC - convert to Central time
    rooftop_pv_cf_cst = (
        rooftop_pv_cf.apply(lambda x: np.roll(x, shift=-6))
        .tz_convert(None)
        .tz_localize('Etc/GMT+6')
    )
    
    return rooftop_pv_cf_cst

In [5]:
rooftop_pv_cf_meta = get_rooftop_pv_cf_meta()
gid_county_map = create_gid_county_map(rooftop_pv_cf_meta)

rooftop_pv_cf_profiles_by_sector = {}
for sector in ['residential', 'commercial']:
    rooftop_pv_cf_profiles_by_sector[sector] = get_rooftop_pv_cf_profile(sector, gid_county_map)

In [8]:
os.makedirs('../data/distpv_profiles/processed', exist_ok=True)
for sector in ['residential', 'commercial']:
    rooftop_pv_cf_profiles_by_sector[sector].to_hdf(
        f'../data/distpv_profiles/processed/county_rooftop_pv_cf_{sector}.h5',
        key='data'
    )